### Preparación del Entorno

In [0]:
# Cargamos la tabla limpia y la convertimos en una Vista Temporal de Spark
df_citas = spark.table("default.citas_pmm_limpioV2")
df_citas.createOrReplaceTempView("v_citas_pmm")

print("Vista 'v_citas_pmm' creada exitosamente. Lista para análisis SQL.")

Vista 'v_citas_pmm' creada exitosamente. Lista para análisis SQL.


### KP1 N1: Rendimiento Financiero y Distribución de Cobertura por Aseguradora

In [0]:
%sql
SELECT 
    nom_aseguradora AS Aseguradora,
    COUNT(*) AS Volumen_Citas,
    ROUND(SUM(pago_aseg), 2) AS Total_Pagado_Aseguradora_USD,
    ROUND(SUM(pago_clie), 2) AS Total_Copago_Paciente_USD,
    ROUND(SUM(pago_total), 2) AS Ingresos_Totales_USD,
    ROUND((SUM(pago_aseg) / SUM(pago_total)) * 100, 2) AS Porcentaje_Cobertura_Promedio
FROM v_citas_pmm
WHERE estado_cita = 'Completada'
GROUP BY nom_aseguradora
ORDER BY Ingresos_Totales_USD DESC;

Aseguradora,Volumen_Citas,Total_Pagado_Aseguradora_USD,Total_Copago_Paciente_USD,Ingresos_Totales_USD,Porcentaje_Cobertura_Promedio
Particular / Sin Seguro,1970,0.0,98130.0,98130.0,0.0
PALIG,1289,45100.53,19434.47,64535.0,69.89
Seguros ASSA,1267,44333.97,18731.03,63065.0,70.3
MAPFRE Panamá,1206,42314.19,18080.81,60395.0,70.06
Compañía Internacional de Seguros (IS),1178,41032.75,17872.25,58905.0,69.66
Seguros SURA,1159,40459.46,17065.54,57525.0,70.33


### KPI N2: Tasa de Cancelación por Sucursal

In [0]:
%sql
SELECT 
    nom_sucursal AS Sucursal,
    COUNT(*) AS Citas_Agendadas,
    SUM(CASE WHEN estado_cita = 'Cancelada' THEN 1 ELSE 0 END) AS Citas_Canceladas,
    SUM(CASE WHEN estado_cita = 'Completada' THEN 1 ELSE 0 END) AS Citas_Completadas,
    ROUND((SUM(CASE WHEN estado_cita = 'Cancelada' THEN 1 ELSE 0 END) / COUNT(*)) * 100, 2) AS Tasa_Cancelacion_PCT
FROM v_citas_pmm
GROUP BY nom_sucursal
ORDER BY Tasa_Cancelacion_PCT DESC;

Sucursal,Citas_Agendadas,Citas_Canceladas,Citas_Completadas,Tasa_Cancelacion_PCT
PMM Brisas del Golf,1452,307,1145,21.14
PMM El Dorado,2019,411,1608,20.36
PMM Tocumen,1018,195,823,19.16
PMM San Francisco,2990,564,2426,18.86
PMM Costa del Este,2521,454,2067,18.01


### KPI N3: Segmentación Demográfica y Ticket Promedio

In [0]:
%sql
SELECT 
    CASE 
        WHEN edad_pac_cita < 15 THEN '1. Pediátrico (<15 años)'
        WHEN edad_pac_cita BETWEEN 15 AND 64 THEN '2. Adulto (15-64 años)'
        ELSE '3. Geriátrico (65+ años)' 
    END AS Segmento_Poblacional,
    COUNT(*) AS Total_Atenciones,
    ROUND(SUM(pago_total), 2) AS Facturacion_Total_USD,
    ROUND(AVG(pago_total), 2) AS Ticket_Promedio_USD
FROM v_citas_pmm
WHERE estado_cita = 'Completada'
GROUP BY 
    CASE 
        WHEN edad_pac_cita < 15 THEN '1. Pediátrico (<15 años)'
        WHEN edad_pac_cita BETWEEN 15 AND 64 THEN '2. Adulto (15-64 años)'
        ELSE '3. Geriátrico (65+ años)' 
    END
ORDER BY Segmento_Poblacional ASC;

Segmento_Poblacional,Total_Atenciones,Facturacion_Total_USD,Ticket_Promedio_USD
1. Pediátrico (<15 años),1636,69475.0,42.47
2. Adulto (15-64 años),4717,243460.0,51.61
3. Geriátrico (65+ años),1716,89620.0,52.23


### KPI N4: Rentabilidad de cada Especialidad por Hora

In [0]:
%sql
SELECT 
    especialidad_medica AS Especialidad,
    COUNT(*) AS Total_Consultas,
    ROUND(SUM(mins_cit) / 60, 2) AS Horas_Totales_Invertidas,
    ROUND(SUM(pago_total), 2) AS Ingresos_Totales_USD,
    ROUND(SUM(pago_total) / (SUM(mins_cit) / 60), 2) AS Rentabilidad_Por_Hora_USD
FROM v_citas_pmm
WHERE estado_cita = 'Completada'
GROUP BY especialidad_medica
ORDER BY Rentabilidad_Por_Hora_USD DESC

Especialidad,Total_Consultas,Horas_Totales_Invertidas,Ingresos_Totales_USD,Rentabilidad_Por_Hora_USD
Neurología,468,287.0,35100.0,122.3
Cardiología,380,233.25,26600.0,114.04
Nefrología,469,293.5,32830.0,111.86
Gastroenterología,457,279.75,29705.0,106.18
Psiquiatría,444,275.0,26640.0,96.87
Radiología,679,425.75,37345.0,87.72
Dermatología,710,434.25,35500.0,81.75
Otorrinolaringología,692,435.5,34600.0,79.45
Geriatría,137,89.25,6850.0,76.75
Ginecología,281,175.0,12645.0,72.26


### KPI N5: Análisis de Citas por Jornada y Hora

In [0]:
%sql
SELECT 
    CASE 
        -- REGEXP_EXTRACT busca la primera coincidencia de hora (ej: "09:30") y extrae solo los primeros 2 dígitos (\d{2})
        WHEN CAST(REGEXP_EXTRACT(hr_inicio_cit, '(\\d{2}):\\d{2}', 1) AS INT) < 12 THEN 'Mañana (07:00 - 11:59)'
        WHEN CAST(REGEXP_EXTRACT(hr_inicio_cit, '(\\d{2}):\\d{2}', 1) AS INT) BETWEEN 12 AND 16 THEN 'Tarde (12:00 - 16:59)'
        ELSE 'Noche (17:00 - 20:00)'
    END AS Jornada_Horaria,
    COUNT(*) AS Cantidad_Citas,
    ROUND((COUNT(*) / (SELECT COUNT(*) FROM v_citas_pmm)) * 100, 2) AS Porcentaje_Demanda_PCT
FROM v_citas_pmm
GROUP BY 
    CASE 
        WHEN CAST(REGEXP_EXTRACT(hr_inicio_cit, '(\\d{2}):\\d{2}', 1) AS INT) < 12 THEN 'Mañana (07:00 - 11:59)'
        WHEN CAST(REGEXP_EXTRACT(hr_inicio_cit, '(\\d{2}):\\d{2}', 1) AS INT) BETWEEN 12 AND 16 THEN 'Tarde (12:00 - 16:59)'
        ELSE 'Noche (17:00 - 20:00)'
    END
ORDER BY Cantidad_Citas DESC;

Jornada_Horaria,Cantidad_Citas,Porcentaje_Demanda_PCT
Tarde (12:00 - 16:59),4553,45.53
Mañana (07:00 - 11:59),4454,44.54
Noche (17:00 - 20:00),993,9.93


### KPI N6: Reembolso Promedio de Aseguradoras por Especialidad

In [0]:
%sql
SELECT 
    especialidad_medica AS Especialidad,
    COUNT(*) AS Citas_Aseguradas,
    ROUND(SUM(pago_aseg), 2) AS Facturacion_A_Seguros_USD,
    ROUND(AVG(pago_aseg), 2) AS Reembolso_Promedio_Aseguradora_USD
FROM v_citas_pmm
WHERE estado_cita = 'Completada' AND nom_aseguradora != 'Sin Seguro'
GROUP BY especialidad_medica
ORDER BY Reembolso_Promedio_Aseguradora_USD DESC;

Especialidad,Citas_Aseguradas,Facturacion_A_Seguros_USD,Reembolso_Promedio_Aseguradora_USD
Neurología,468,18215.41,38.92
Nefrología,469,18238.46,38.89
Gastroenterología,457,16010.52,35.03
Cardiología,380,12929.43,34.02
Psiquiatría,444,13409.49,30.2
Radiología,679,20420.61,30.07
Dermatología,710,19107.78,26.91
Otorrinolaringología,692,18307.43,26.46
Geriatría,137,3597.9,26.26
Urología,266,6896.37,25.93
